# Batch Processing

Process multiple point cloud files with progress tracking and result aggregation.

## When to batch

- **Many files**: Processing dozens or hundreds of tiles
- **Memory management**: Large files benefit from sequential processing to avoid GPU OOM
- **Checkpointing**: Resume processing if interrupted

The batch script processes files in configurable chunks (default: 10 per batch).

In [ ]:
# Survey input files
from pathlib import Path
import os

input_dir = Path('/data/input')
output_dir = Path('/data/output')

files = sorted(
    list(input_dir.glob('*.las')) +
    list(input_dir.glob('*.laz')) +
    list(input_dir.glob('*.ply'))
)

total_size = sum(f.stat().st_size for f in files)
print(f'Input files: {len(files)}')
print(f'Total size:  {total_size / 1e9:.2f} GB')
print()
for f in files:
    print(f'  {f.name:40s} {f.stat().st_size / 1e6:8.1f} MB')

## Run Batch Inference

The batch script splits files into chunks and processes each chunk sequentially.
Adjust the batch size based on your GPU memory and file sizes.

In [ ]:
import subprocess, os

os.environ['SAT_ROOT'] = os.environ.get('SAT_ROOT', '/opt/segmentanytree')
BATCH_SIZE = 10  # Files per batch — reduce if GPU memory is limited

result = subprocess.run(
    ['bash', 'scripts/run_batch_inference.sh',
     str(input_dir), str(output_dir), str(BATCH_SIZE)],
    cwd=os.environ['SAT_ROOT'],
    capture_output=False,
    text=True
)
print(f'Exit code: {result.returncode}')

## Monitor Progress

In [ ]:
results_dir = output_dir / 'final_results'
if results_dir.exists():
    outputs = list(results_dir.glob('*.copc.laz')) + list(results_dir.glob('*.laz')) + list(results_dir.glob('*.las'))
    print(f'Progress: {len(outputs)} / {len(files)} files processed')
    print(f'Output size: {sum(f.stat().st_size for f in outputs) / 1e9:.2f} GB')
    print()
    for f in sorted(outputs):
        print(f'  {f.name:50s} {f.stat().st_size / 1e6:8.1f} MB')
else:
    print('No results yet — inference may still be running.')

## Aggregate Results

In [ ]:
import laspy
import numpy as np
import pandas as pd

results_dir = output_dir / 'final_results'
result_files = sorted(results_dir.glob('*.copc.laz')) or sorted(results_dir.glob('*.laz')) or sorted(results_dir.glob('*.las'))

summary = []
for f in result_files:
    las = laspy.read(str(f))
    n_points = len(las.points)

    n_trees = 0
    if 'PredInstance' in las.point_format.dimension_names:
        tree_ids = np.unique(las.PredInstance)
        n_trees = len(tree_ids[tree_ids > 0])

    n_tree_points = 0
    if 'PredSemantic' in las.point_format.dimension_names:
        n_tree_points = int((las.PredSemantic == 2).sum())

    summary.append({
        'file': f.name,
        'points': n_points,
        'trees': n_trees,
        'tree_points': n_tree_points,
        'tree_pct': round(100 * n_tree_points / n_points, 1) if n_points > 0 else 0,
        'size_mb': round(f.stat().st_size / 1e6, 1),
    })

df = pd.DataFrame(summary)
print(f'Total: {len(df)} files, {df["trees"].sum()} trees, {df["points"].sum():,} points')
print()
print(df.to_string(index=False))

## Multi-GPU Processing

For systems with multiple GPUs, the parallel inference script distributes files across GPUs:

```bash
# Auto-detect all GPUs
bash scripts/run_inference_parallel.sh /data/input /data/output

# Specify GPU count
bash scripts/run_inference_parallel.sh /data/input /data/output 4
```

Files are assigned round-robin. Each GPU runs an independent pipeline. Results are merged into `output/final_results/`.

See the [Inference Guide](../docs/inference.md) for details.